# SEC EDGAR Filings Scraper

Scrapes SEC EDGAR filings by company CIK number, truncates filing data to concise summaries, and displays results organized by company name.

**Run all cells in order**, or just edit the config cell and hit **Runtime > Run all**.

In [ ]:
#@title Configuration { display-mode: "form" }

#@markdown ### Companies to scrape (CIK numbers)
#@markdown Common CIKs: Apple=320193, Microsoft=789019, Tesla=1318605, Amazon=1018724, Google=1652044, Meta=1326801, NVIDIA=1045810
CIKS = "320193, 789019, 1318605" #@param {type:"string"}

#@markdown ### Filing types to include
FORM_TYPES = "10-K, 10-Q, 8-K" #@param {type:"string"}

#@markdown ### How many days back to search
DAYS_BACK = 365 #@param {type:"slider", min:1, max:365, step:1}

#@markdown ### Fetch and truncate actual filing content?
FETCH_CONTENT = False #@param {type:"boolean"}

#@markdown ### Max characters for truncated content
MAX_CONTENT_LENGTH = 500 #@param {type:"integer"}

#@markdown ### Your contact info (SEC requires this)
USER_AGENT = "EdgarScraper/1.0 (your-email@example.com)" #@param {type:"string"}

# Parse comma-separated inputs
cik_list = [c.strip() for c in CIKS.split(",") if c.strip()]
form_list = [f.strip() for f in FORM_TYPES.split(",") if f.strip()]

print(f"CIKs: {cik_list}")
print(f"Forms: {form_list}")
print(f"Days back: {DAYS_BACK}")

In [ ]:
#@title Scraper Functions (run this cell)

import csv
import json
import os
import re
import time
from datetime import datetime, timedelta

import requests
import pandas as pd

EDGAR_SUBMISSIONS_URL = "https://data.sec.gov/submissions"


def get_headers(user_agent):
    return {"User-Agent": user_agent, "Accept": "application/json"}


def fetch_company_filings(cik, filing_types, user_agent):
    cik_padded = cik.zfill(10)
    url = f"{EDGAR_SUBMISSIONS_URL}/CIK{cik_padded}.json"
    resp = requests.get(url, headers=get_headers(user_agent), timeout=30)
    resp.raise_for_status()
    data = resp.json()

    company_name = data.get("name", "Unknown")
    recent = data.get("filings", {}).get("recent", {})

    forms = recent.get("form", [])
    dates = recent.get("filingDate", [])
    accessions = recent.get("accessionNumber", [])
    primary_docs = recent.get("primaryDocument", [])
    descriptions = recent.get("primaryDocDescription", [])

    filings = []
    for i, form in enumerate(forms):
        if filing_types and form not in filing_types:
            continue
        accession_clean = accessions[i].replace("-", "")
        filings.append({
            "company_name": company_name,
            "cik": cik,
            "form_type": form,
            "filing_date": dates[i],
            "accession_number": accessions[i],
            "primary_document": primary_docs[i] if i < len(primary_docs) else "",
            "description": descriptions[i] if i < len(descriptions) else "",
            "filing_url": (
                f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_clean}/"
                f"{primary_docs[i]}"
                if i < len(primary_docs) else ""
            ),
        })
    return filings


def fetch_filing_content(filing_url, user_agent):
    try:
        resp = requests.get(filing_url, headers=get_headers(user_agent), timeout=60)
        resp.raise_for_status()
        return resp.text
    except Exception as e:
        return f"[Error fetching content: {e}]"


def truncate_content(text, max_length=500):
    if not text:
        return ""
    clean = re.sub(r"<[^>]+>", " ", text)
    clean = re.sub(r"\\s+", " ", clean).strip()
    return clean[:max_length] + "..." if len(clean) > max_length else clean


def scrape_filings(ciks, filing_types, days_back, user_agent,
                    fetch_content=False, max_content_length=500):
    date_to = datetime.now().strftime("%Y-%m-%d")
    date_from = (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d")
    all_filings = []

    for cik in ciks:
        print(f"Fetching filings for CIK {cik}...")
        try:
            filings = fetch_company_filings(cik, filing_types, user_agent)
        except Exception as e:
            print(f"  Error fetching CIK {cik}: {e}")
            continue

        filings = [f for f in filings if date_from <= f.get("filing_date", "") <= date_to]

        if fetch_content:
            for filing in filings:
                url = filing.get("filing_url", "")
                if url:
                    print(f"  Fetching content: {filing['form_type']} ({filing['filing_date']})...")
                    raw = fetch_filing_content(url, user_agent)
                    filing["content_truncated"] = truncate_content(raw, max_content_length)
                    time.sleep(0.1)

        for filing in filings:
            filing["scraped_at"] = datetime.now().isoformat()

        all_filings.extend(filings)
        print(f"  Found {len(filings)} filings in date range.")
        time.sleep(0.1)

    return all_filings


print("Scraper functions loaded.")

In [ ]:
#@title Run the Scraper

print("=" * 60)
print("SEC EDGAR Filings Scraper")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"CIKs: {', '.join(cik_list)}")
print(f"Forms: {', '.join(form_list)}")
print(f"Days back: {DAYS_BACK}")
print("=" * 60)

filings = scrape_filings(
    ciks=cik_list,
    filing_types=form_list,
    days_back=DAYS_BACK,
    user_agent=USER_AGENT,
    fetch_content=FETCH_CONTENT,
    max_content_length=MAX_CONTENT_LENGTH,
)

if filings:
    df = pd.DataFrame(filings)
    print(f"\nTotal filings found: {len(df)}")
    print(f"Companies: {df['company_name'].nunique()}")
    print(f"Date range: {df['filing_date'].min()} to {df['filing_date'].max()}")
else:
    print("\nNo filings found for the given criteria.")
    df = pd.DataFrame()

In [ ]:
#@title View Results by Company

if not df.empty:
    for company, group in df.groupby("company_name"):
        print(f"\n{'=' * 60}")
        print(f"{company} (CIK: {group['cik'].iloc[0]})")
        print(f"{'=' * 60}")
        display(group[["form_type", "filing_date", "description", "filing_url"]].reset_index(drop=True))
else:
    print("No data to display. Run the scraper first.")

In [ ]:
#@title Save & Download Files

if not df.empty:
    date_stamp = datetime.now().strftime("%Y%m%d")
    output_dir = "edgar_data"
    os.makedirs(output_dir, exist_ok=True)

    # Save combined files
    json_path = os.path.join(output_dir, f"filings_{date_stamp}.json")
    csv_path = os.path.join(output_dir, f"filings_{date_stamp}.csv")

    df.to_json(json_path, orient="records", indent=2)
    df.to_csv(csv_path, index=False)
    print(f"Saved {len(df)} filings to {json_path}")
    print(f"Saved {len(df)} filings to {csv_path}")

    # Save per-company files
    company_dir = os.path.join(output_dir, "by_company")
    os.makedirs(company_dir, exist_ok=True)
    for company, group in df.groupby("company_name"):
        safe_name = re.sub(r"[^\w\s-]", "", company).strip().replace(" ", "_")
        company_path = os.path.join(company_dir, f"{safe_name}_{date_stamp}.json")
        group.to_json(company_path, orient="records", indent=2)
        print(f"Saved {len(group)} filings to {company_path}")

    # Download in Colab
    try:
        from google.colab import files
        print("\nDownloading CSV...")
        files.download(csv_path)
    except ImportError:
        print("\nNot running in Colab - files saved locally.")
else:
    print("No data to save. Run the scraper first.")

In [ ]:
#@title Filing Type Breakdown (Chart)

if not df.empty:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Filings by type
    df["form_type"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
    axes[0].set_title("Filings by Type")
    axes[0].set_ylabel("Count")

    # Filings by company
    df["company_name"].value_counts().plot(kind="barh", ax=axes[1], color="coral")
    axes[1].set_title("Filings by Company")
    axes[1].set_xlabel("Count")

    plt.tight_layout()
    plt.show()
else:
    print("No data to chart. Run the scraper first.")